# 01 – Quickstart RAG with LlamaIndex

In this notebook we will:
1. Load the Paul Graham *What I Worked On* essay using `SimpleDirectoryReader`.
2. Build a `VectorStoreIndex` from the loaded documents.
3. Create a query engine and run a basic question-answering query.

> **Requirements**: Make sure you have run the `postBuild` script (or executed the download cell below) so that `data/paul_graham_essay.txt` exists.

## 0. Install dependencies

Uncomment the cell below if you are running outside of Binder and have not yet installed the requirements.

In [1]:
# %pip install llama-index openai python-dotenv ipywidgets -q

## 1. Set your OpenAI API Key

Enter your API key below. It will **not** be stored in the notebook.

In [2]:
import os
from getpass import getpass

# Enter your OpenAI API key when prompted.
# The key is stored only in memory for this session and is never saved to disk.
openai_api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = openai_api_key

## 2. (Optional) Download the essay

If you are not on Binder (where the `postBuild` script runs automatically), run the cell below to download the data.

In [3]:
import os
import re
import html
import urllib.request

os.makedirs("data", exist_ok=True)

essay_path = "data/paul_graham_essay.txt"
if not os.path.exists(essay_path):
    url = "https://paulgraham.com/worked.html"
    with urllib.request.urlopen(url) as response:
        page_html = response.read().decode("utf-8", errors="ignore")

    # Convert simple HTML to plain text for indexing.
    text = re.sub(r"<script[\\s\\S]*?</script>", " ", page_html, flags=re.IGNORECASE)
    text = re.sub(r"<style[\\s\\S]*?</style>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", text)
    text = html.unescape(text)
    text = re.sub(r"\\s+", " ", text).strip()

    with open(essay_path, "w", encoding="utf-8") as f:
        f.write(text)

    print(f"Downloaded and converted essay to {essay_path}")
else:
    print(f"Essay already exists at {essay_path}")

Essay already exists at data/paul_graham_essay.txt


## 3. Load the documents

In [4]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader("data").load_data()

print(f"Loaded {len(documents)} document(s)")
print(f"First 200 characters:\n{documents[0].text[:200]}")

Loaded 1 document(s)
First 200 characters:
What I Worked On   -->
 
                                          February 2021  Before college the two main things I worked on, outside of school,
were writing and programming. I didn't write essays


## 4. Build the VectorStoreIndex

In [5]:
from llama_index.core import VectorStoreIndex

# Building the index will:
#   1. Split each document into chunks (nodes).
#   2. Generate an embedding for every chunk using OpenAI's embedding model.
#   3. Store the embeddings in an in-memory vector store.
index = VectorStoreIndex.from_documents(documents)

print("Index built successfully!")

Index built successfully!


## 5. Create a query engine and ask a question

In [6]:
query_engine = index.as_query_engine()

response = query_engine.query(
    "What did Paul Graham work on before Y Combinator?"
)

print(response)

Before Y Combinator, Paul Graham worked on projects with Robert and Trevor, and he also considered angel investing.


### Try your own question

In [9]:
your_question = input("Ask a question about the essay: ")
response = query_engine.query(your_question)
print(response)

He started taking art classes at Harvard while in a PhD program in computer science. Later, he applied to art schools and was accepted into the BFA program at RISD. Additionally, he took the entrance exam at the Accademia di Belli Arti in Florence.
